In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
import bayesflow as bf
import numpy as np
import matplotlib.pyplot as plt

# Simple example

## Simulator

In [ ]:
def context(batch_size, n=None):
    if n is None:
        n = np.random.randint(10, 30)

    return dict(n=n)

def prior(mu=None, sigma=None):
    if mu is None:
        mu = np.random.normal(loc=0.0, scale=1.0)
    if sigma is None:
        sigma = np.random.gamma(shape=4.0, scale=0.1)

    return dict(mu=mu, sigma=sigma)

def likelihood(n, mu, sigma):
    y = np.random.normal(mu, sigma, size=n)

    return dict(y=y)

def summary(y):
    mean = np.mean(y)
    sd = np.std(y, ddof=1)
    
    return dict(mean=mean, sd=sd)

In [ ]:
simulator = bf.make_simulator([prior, likelihood, summary], meta_fn=context)

### Prior predictives

In [ ]:
data = simulator.sample(1000)

In [ ]:
fig=bf.diagnostics.pairs_samples(data, variable_keys=["mean", "sd"])

## Approximator

In [ ]:
adapter = (bf.Adapter()
    .broadcast("n", to="mean")
    .constrain("sigma", lower=0)
    .concatenate(["n", "mean", "sd"], into="inference_conditions")
    .concatenate(["mu", "sigma"], into="inference_variables")
    .drop("y")
    )

In [ ]:
adapter(data)

In [ ]:
inference_network=bf.networks.CouplingFlow(transform="spline")

In [ ]:
workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    inference_network=inference_network,
)

In [ ]:
epochs=10
num_batches=100
batch_size=256

In [ ]:
history = workflow.fit_online(
    epochs=epochs,
    num_batches=num_batches,
    batch_size=batch_size,
)

In [ ]:
fig=bf.diagnostics.plots.loss(history)

## Validation

## Plot default diagnostics

In [ ]:
test_data = simulator.sample(1000, n=20)
workflow.plot_default_diagnostics(test_data)

### Run some diagnostics manually

In [ ]:
prior = dict(mu=test_data["mu"], sigma=test_data["sigma"])
posterior = workflow.sample(num_samples=500, conditions=test_data)

In [ ]:
fig=bf.diagnostics.plots.calibration_ecdf(estimates=posterior, targets=prior)

In [ ]:
fig=bf.diagnostics.z_score_contraction(estimates=posterior, targets=prior)

In [ ]:
fig=bf.diagnostics.plots.recovery(estimates=posterior, targets=prior)

## Inference

In [ ]:
inference_data = dict(n=20, mean=np.array([[-0.5]]), sd=np.array([[0.7]]))

In [ ]:
posterior = workflow.sample(num_samples=1000, conditions=inference_data)

In [ ]:
fig=bf.diagnostics.pairs_posterior(estimates=posterior, priors=prior)

# Using a summary network

## Define and run the workflow

In [ ]:
adapter = (bf.Adapter()
    .broadcast("n", to="y")
    .as_set("y")
    .constrain("sigma", lower=0)
    .rename("n", "inference_conditions")
    .rename("y", "summary_variables")
    .concatenate(["mu", "sigma"], into="inference_variables")
    .drop(["mean", "sd"])
    )

In [ ]:
workflow = bf.BasicWorkflow(
    inference_network=bf.networks.CouplingFlow(transform="spline"), 
    summary_network=bf.networks.DeepSet(),
    simulator=simulator,
    adapter=adapter,
)

In [ ]:
history=workflow.fit_online(
    epochs=epochs, 
    num_batches_per_epoch=num_batches, 
    batch_size=batch_size
)

## Validation

In [ ]:
test_data = simulator.sample(1000, n=20)

In [ ]:
plots=workflow.plot_default_diagnostics(test_data=test_data)

## Inference

In [ ]:
inference_data = dict(
    y = np.random.normal(loc=1, scale=0.6, size=(1, 20)),
    n = 20
)

In [ ]:
num_samples=2_000

In [ ]:
posterior=workflow.sample(num_samples=num_samples, conditions=inference_data)
posterior=keras.tree.map_structure(np.squeeze, posterior)

In [ ]:
fig=bf.diagnostics.pairs_posterior(posterior)

### Posterior Predictive and Out-of-Distribution Checks

In [ ]:
posterior_predictives = simulator.sample(num_samples, n=20, **posterior)

In [ ]:
summary(inference_data["y"])

In [ ]:
fig=bf.diagnostics.pairs_samples(posterior_predictives, variable_keys=["mean", "sd"])

In [ ]:
fig=plt.violinplot(posterior_predictives["y"], showmeans=True, side="low")
fig=plt.scatter(x=[i+1 for i in range(inference_data["n"])], y=inference_data["y"], c="black", zorder=100)

In [ ]:
summaries_null=workflow.summary_network(workflow.simulate_adapted(1000)['summary_variables'])
summaries_ref=workflow.summary_network(workflow.simulate_adapted(500)['summary_variables'], training=False)

In [ ]:
mmd_null = [
    bf.metrics.functional.maximum_mean_discrepancy(summaries_null, summaries_ref[i:i+1]).numpy() for i in range(500)
]

In [ ]:
summaries_obs=workflow.summary_network(adapter(inference_data, strict=False)["summary_variables"])
mmd_obs=bf.metrics.functional.maximum_mean_discrepancy(summaries_null, summaries_obs)

In [ ]:
fig=bf.diagnostics.plots.mmd_hypothesis_test(mmd_null, mmd_obs)

# Further Tasks

1. Train a BayesFlow model that estimates the mean vector and a variance-covariance matrix of a 2D Gaussian.
2. Train a BayesFlow model that estimates parameter of any other distribution. You are free to choose which (some examples: Gamma, Negative binomial, Weibull,...)
3. The implementation of the normal model imposes one particular configuration of the priors on the parameters `"mu"` and `"sigma"`. This is relatively impractical, because not every time such priors would be reasonable. It is also relatively common to fit a model with different priors to investigate how robust are your inferences against prior specification. To do this, it is handy if you train a single BayesFlow model that can make inferences with different prior specifications. You can do this by varying the prior specification during simulations. For example,
   
```{.python}
hyper_mu_mu = np.random.uniform(-100, 100)
mu = np.random.normal(hyper_mu_mu)
```

would generate different priors for mu, depending on the value of the hyperparameter $\mu_\mu$. If you condition the network on the values of the hyperparameters, you can train the networks to be able to use different priors during inference. Try to implement such a network.